# 🫀 Sistema de Detección de Arritmias en ECG usando Deep Learning

## Universidad - Proyecto Final de Big Data

### Descripción del Proyecto

Este notebook implementa un sistema de detección de arritmias cardíacas utilizando:
- **Dataset**: MIT-BIH Arrhythmia Database de PhysioNet
- **Modelo**: Autoencoder LSTM para detección de anomalías
- **Aplicación**: Análisis en tiempo real de señales ECG mediante Apache Kafka

### Estructura del Notebook

1. **Configuración del Entorno** - Instalación de dependencias
2. **Descarga de Datos** - Obtención del dataset MIT-BIH
3. **Exploración de Datos** - Análisis y visualización de las señales ECG
4. **Preprocesamiento** - Segmentación y preparación de datos
5. **División Train/Test** - Separación a nivel de paciente (sin data leakage)
6. **Entrenamiento del Modelo** - Autoencoder LSTM
7. **Evaluación** - Métricas y visualización de resultados
8. **Exportación del Modelo** - Guardado para uso en streaming

### Autores
- Trabajo desarrollado para el curso de Big Data

---

## 1. 📦 Configuración del Entorno

Instalamos las bibliotecas necesarias para el procesamiento de señales ECG y machine learning.

In [ ]:
# Instalación de dependencias
!pip install -q wfdb pyarrow torch scikit-learn matplotlib seaborn

In [ ]:
# Importación de librerías
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import wfdb
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# Configuración de estilo para gráficos
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

# Semilla para reproducibilidad
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

print("✅ Librerías importadas correctamente")
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {'CUDA' if torch.cuda.is_available() else 'CPU'}")

## 2. 📥 Descarga de Datos

El dataset **MIT-BIH Arrhythmia Database** contiene 48 registros de ECG de media hora cada uno, con anotaciones de cardiólogos expertos.

### Características del Dataset:
- **Frecuencia de muestreo**: 360 Hz
- **Resolución**: 11 bits sobre rango de 10 mV
- **Canales**: 2 derivaciones (típicamente MLII y V1/V5)
- **Anotaciones**: Tipos de latido marcados por expertos

In [ ]:
# Configuración de rutas
# El notebook puede ejecutarse tanto en Google Colab como localmente

# Detectar si estamos en Colab
try:
    import google.colab
    IN_COLAB = True
    BASE_DIR = '/content'
except ImportError:
    IN_COLAB = False
    BASE_DIR = os.getcwd()

DATA_DIR = os.path.join(BASE_DIR, 'physionet.org/files/mitdb/1.0.0')
OUTPUT_DIR = os.path.join(BASE_DIR, 'models')

# Crear directorio de modelos si no existe
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"📁 Directorio base: {BASE_DIR}")
print(f"📁 Directorio de datos: {DATA_DIR}")
print(f"📁 Directorio de modelos: {OUTPUT_DIR}")

In [ ]:
# Descarga del dataset MIT-BIH desde PhysioNet
# Solo descargar si no existe
if not os.path.exists(DATA_DIR):
    print("⬇️ Descargando dataset MIT-BIH desde PhysioNet...")
    !wget -r -N -c -np -q --show-progress https://physionet.org/files/mitdb/1.0.0/
    print("✅ Descarga completada")
else:
    print("✅ Dataset ya descargado")

## 3. 📊 Carga y Exploración de Datos

Utilizamos la librería `wfdb` para leer los archivos de PhysioNet en su formato nativo (.dat, .hea, .atr).

In [ ]:
def load_mitbih_records(data_dir):
    """
    Carga todos los registros del dataset MIT-BIH.
    
    Args:
        data_dir: Directorio con los archivos del dataset
        
    Returns:
        DataFrame con todas las señales y anotaciones
    """
    # Obtener lista de registros
    files = [f for f in os.listdir(data_dir) if f.endswith('.hea')]
    record_ids = sorted(list(set([f.split('.')[0] for f in files if f[0].isdigit()])))
    
    print(f"📋 Encontrados {len(record_ids)} registros")
    
    all_dataframes = []
    
    for rec_id in record_ids:
        try:
            path_to_record = os.path.join(data_dir, rec_id)
            
            # Leer registro y anotaciones con wfdb
            record = wfdb.rdrecord(path_to_record)
            annotation = wfdb.rdann(path_to_record, 'atr')
            
            # Crear DataFrame con la señal
            df = pd.DataFrame(record.p_signal, columns=record.sig_name)
            
            # Añadir metadatos
            df['record_id'] = rec_id
            df['fs'] = record.fs
            df['sample_idx'] = range(len(df))
            
            # Añadir anotaciones
            df['annotation'] = None
            valid_indices = annotation.sample < len(df)
            safe_samples = annotation.sample[valid_indices]
            safe_symbols = np.array(annotation.symbol)[valid_indices]
            df.loc[safe_samples, 'annotation'] = safe_symbols
            
            # Optimizar tipos de datos
            df['record_id'] = df['record_id'].astype('category')
            
            all_dataframes.append(df)
            print(f"  ✓ Procesado: {rec_id}")
            
        except Exception as e:
            print(f"  ✗ Error en {rec_id}: {e}")
    
    # Concatenar todos los DataFrames
    print("\n🔄 Concatenando registros...")
    full_df = pd.concat(all_dataframes, ignore_index=True)
    
    print(f"✅ Dataset cargado: {len(full_df):,} muestras totales")
    return full_df

# Cargar datos
df_completo = load_mitbih_records(DATA_DIR)

In [ ]:
# Exploración básica del dataset
print("\n" + "="*60)
print("📊 INFORMACIÓN DEL DATASET")
print("="*60)
print(f"\nColumnas: {list(df_completo.columns)}")
print(f"\nTotal de muestras: {len(df_completo):,}")
print(f"Número de pacientes: {df_completo['record_id'].nunique()}")
print(f"Frecuencia de muestreo: {df_completo['fs'].iloc[0]} Hz")
print(f"\nDistribución de anotaciones:")
print(df_completo['annotation'].value_counts())

In [ ]:
# Visualización de una señal ECG de ejemplo
def plot_ecg_segment(df, record_id, start_sec=0, duration_sec=10):
    """
    Visualiza un segmento de señal ECG con sus anotaciones.
    """
    fs = 360  # Hz
    start_sample = int(start_sec * fs)
    end_sample = int((start_sec + duration_sec) * fs)
    
    df_patient = df[df['record_id'] == record_id].copy()
    df_patient.reset_index(drop=True, inplace=True)
    
    segment = df_patient.iloc[start_sample:end_sample]
    
    # Obtener nombre de la columna de señal
    signal_cols = [c for c in segment.columns if c not in ['record_id', 'fs', 'sample_idx', 'annotation']]
    signal_col = signal_cols[0] if signal_cols else None
    
    if signal_col is None:
        print("No se encontró columna de señal")
        return
    
    fig, ax = plt.subplots(figsize=(14, 4))
    
    time = np.arange(len(segment)) / fs + start_sec
    ax.plot(time, segment[signal_col].values, 'b-', linewidth=0.8, label=f'Signal ({signal_col})')
    
    # Marcar anotaciones
    annotations = segment.dropna(subset=['annotation'])
    if len(annotations) > 0:
        ann_time = annotations.index / fs + start_sec
        ax.scatter(ann_time, annotations[signal_col], c='red', marker='x', s=50, zorder=5)
        for idx, row in annotations.iterrows():
            ax.annotate(row['annotation'], 
                       (idx/fs + start_sec, row[signal_col]), 
                       xytext=(0, 15), textcoords='offset points',
                       color='red', fontsize=8, ha='center')
    
    ax.set_title(f'📈 ECG del Paciente {record_id} (segundos {start_sec}-{start_sec+duration_sec})', fontsize=12)
    ax.set_xlabel('Tiempo (segundos)')
    ax.set_ylabel('Amplitud (mV)')
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# Mostrar ejemplo
plot_ecg_segment(df_completo, '100', start_sec=0, duration_sec=10)

## 4. 🔧 Preprocesamiento de Datos

### Estrategia de Segmentación

Para el modelo de forecasting, segmentamos la señal en ventanas centradas en cada latido:
- **X (entrada)**: 4 segundos de historia antes del pico R (1440 muestras)
- **Y (salida)**: 0.5 segundos después del pico R (180 muestras)

El modelo aprenderá a predecir el segmento futuro a partir del pasado. Las anomalías se detectan por la diferencia entre la predicción y la realidad (error de reconstrucción).

In [ ]:
def segment_beats(df, fs=360, window_past_sec=4, window_future_sec=0.5):
    """
    Segmenta la señal ECG en ventanas centradas en cada latido.
    
    Args:
        df: DataFrame con la señal y anotaciones
        fs: Frecuencia de muestreo
        window_past_sec: Segundos de historia
        window_future_sec: Segundos a predecir
        
    Returns:
        DataFrame con X_past, Y_future, label, record_id
    """
    WINDOW_PAST = int(fs * window_past_sec)
    WINDOW_FUTURE = int(fs * window_future_sec)
    TOTAL_LEN = WINDOW_PAST + WINDOW_FUTURE
    
    segments = []
    
    for record_id in df['record_id'].unique():
        df_patient = df[df['record_id'] == record_id].copy()
        df_patient.reset_index(drop=True, inplace=True)
        
        # Obtener columna de señal
        signal_cols = [c for c in df_patient.columns if c not in ['record_id', 'fs', 'sample_idx', 'annotation']]
        if not signal_cols:
            continue
        signal_col = signal_cols[0]
        signal = df_patient[signal_col].values
        
        # Obtener posiciones de latidos
        beat_indices = df_patient.dropna(subset=['annotation']).index.tolist()
        beat_labels = df_patient.loc[beat_indices, 'annotation'].tolist()
        
        for idx, label in zip(beat_indices, beat_labels):
            start_idx = idx - WINDOW_PAST + 1
            end_idx = idx + WINDOW_FUTURE + 1
            
            if start_idx < 0 or end_idx > len(signal):
                continue
            
            full_window = signal[start_idx:end_idx]
            
            if len(full_window) == TOTAL_LEN:
                X_past = full_window[:WINDOW_PAST].tolist()
                Y_future = full_window[WINDOW_PAST:].tolist()
                
                segments.append({
                    'record_id': record_id,
                    'r_index': idx,
                    'label': label,
                    'X_past': X_past,
                    'Y_future': Y_future
                })
    
    df_segments = pd.DataFrame(segments)
    print(f"✅ Segmentación completada: {len(df_segments):,} latidos")
    return df_segments

# Segmentar datos
df_segmented = segment_beats(df_completo)

In [ ]:
# Visualización de distribución de clases
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Conteo de clases
class_counts = df_segmented['label'].value_counts()

# Gráfico de barras
axes[0].bar(class_counts.index, class_counts.values, color=plt.cm.viridis(np.linspace(0, 1, len(class_counts))))
axes[0].set_title('📊 Distribución de Tipos de Latidos', fontsize=12)
axes[0].set_xlabel('Tipo de Latido')
axes[0].set_ylabel('Cantidad')
axes[0].tick_params(axis='x', rotation=45)

# Normal vs Anormal
normal_count = class_counts.get('N', 0)
abnormal_count = class_counts.sum() - normal_count
axes[1].pie([normal_count, abnormal_count], 
           labels=['Normal (N)', 'Arritmias'], 
           autopct='%1.1f%%',
           colors=['#2ecc71', '#e74c3c'],
           explode=(0, 0.05))
axes[1].set_title('📊 Normal vs Arritmias', fontsize=12)

plt.tight_layout()
plt.show()

print(f"\n📈 Estadísticas:")
print(f"   Latidos normales (N): {normal_count:,}")
print(f"   Arritmias: {abnormal_count:,}")
print(f"   Ratio de desbalance: {normal_count/max(abnormal_count, 1):.2f}:1")

## 5. ✂️ División Train/Test (Sin Data Leakage)

### ⚠️ Importante: Prevención de Data Leakage

Para evitar data leakage (fuga de datos), realizamos la división a **nivel de paciente**, no a nivel de muestra. Esto significa:

- Los pacientes de entrenamiento son **completamente diferentes** de los pacientes de test
- No hay ningún latido del mismo paciente tanto en train como en test
- Esta estrategia refleja mejor el rendimiento real del modelo en pacientes nuevos

### Estrategia de División:
- **80%** de los pacientes → Entrenamiento (solo latidos normales)
- **20%** de los pacientes → Test (normales + arritmias)

In [ ]:
def patient_level_split(df, test_ratio=0.2, random_seed=42):
    """
    Divide los datos a nivel de paciente para evitar data leakage.
    
    Args:
        df: DataFrame con los segmentos
        test_ratio: Proporción de pacientes para test
        random_seed: Semilla para reproducibilidad
        
    Returns:
        df_train (solo normales), df_test (todos los tipos)
    """
    np.random.seed(random_seed)
    
    # Obtener lista de pacientes únicos
    all_patients = df['record_id'].unique()
    n_patients = len(all_patients)
    n_test = max(1, int(n_patients * test_ratio))
    
    # Mezclar y dividir
    shuffled_patients = np.random.permutation(all_patients)
    test_patients = shuffled_patients[:n_test]
    train_patients = shuffled_patients[n_test:]
    
    print(f"📊 División de pacientes:")
    print(f"   Train: {len(train_patients)} pacientes")
    print(f"   Test: {len(test_patients)} pacientes")
    print(f"\n   Pacientes de Train: {list(train_patients[:5])}... ({len(train_patients)} total)")
    print(f"   Pacientes de Test: {list(test_patients)}")
    
    # Filtrar DataFrames
    df_train_all = df[df['record_id'].isin(train_patients)]
    df_test = df[df['record_id'].isin(test_patients)]
    
    # Para entrenamiento del autoencoder, solo usamos latidos NORMALES
    df_train = df_train_all[df_train_all['label'] == 'N'].copy()
    
    print(f"\n📈 Tamaño de conjuntos:")
    print(f"   Train (solo N): {len(df_train):,} latidos")
    print(f"   Test (todos): {len(df_test):,} latidos")
    print(f"   - Normales en test: {len(df_test[df_test['label'] == 'N']):,}")
    print(f"   - Arritmias en test: {len(df_test[df_test['label'] != 'N']):,}")
    
    return df_train, df_test, train_patients, test_patients

# Realizar división
df_train, df_test, train_patients, test_patients = patient_level_split(df_segmented, test_ratio=0.2)

In [ ]:
# Preparar datos para PyTorch
print("🔄 Preparando tensores...")

# Convertir a arrays numpy
X_train = np.array(df_train['X_past'].tolist()).astype(np.float32)
y_train = np.array(df_train['Y_future'].tolist()).astype(np.float32)

X_test = np.array(df_test['X_past'].tolist()).astype(np.float32)
y_test = np.array(df_test['Y_future'].tolist()).astype(np.float32)
labels_test = np.array(df_test['label'].tolist())

print(f"   X_train shape: {X_train.shape}")
print(f"   y_train shape: {y_train.shape}")
print(f"   X_test shape: {X_test.shape}")
print(f"   y_test shape: {y_test.shape}")

# Normalización usando SOLO datos de entrenamiento
print("\n📏 Normalizando datos...")
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

# Ajustar scalers SOLO con datos de train (evitar data leakage)
scaler_X.fit(X_train.reshape(-1, 1))
scaler_y.fit(y_train.reshape(-1, 1))

# Transformar
X_train_norm = scaler_X.transform(X_train.reshape(-1, 1)).reshape(X_train.shape)
y_train_norm = scaler_y.transform(y_train.reshape(-1, 1)).reshape(y_train.shape)

X_test_norm = scaler_X.transform(X_test.reshape(-1, 1)).reshape(X_test.shape)
y_test_norm = scaler_y.transform(y_test.reshape(-1, 1)).reshape(y_test.shape)

print("✅ Normalización completada")
print(f"   Rango X_train: [{X_train_norm.min():.4f}, {X_train_norm.max():.4f}]")
print(f"   Rango X_test: [{X_test_norm.min():.4f}, {X_test_norm.max():.4f}]")

## 6. 🧠 Modelo: LSTM Autoencoder

### Arquitectura del Modelo

Utilizamos un **Autoencoder basado en LSTM** para detección de anomalías:

1. **Encoder**: Comprime la secuencia de entrada en un vector latente
2. **Decoder**: Reconstruye la secuencia futura a partir del vector latente

### Principio de Detección de Anomalías:

- El modelo se entrena **solo con latidos normales**
- Aprende a predecir el patrón futuro de un ECG normal
- Cuando encuentra una arritmia, el error de predicción será mayor
- Un umbral en el error permite clasificar normal vs anomalía

In [ ]:
class LSTMAutoencoder(nn.Module):
    """
    LSTM Autoencoder para predicción de series temporales ECG.
    """
    def __init__(self, input_size=1, hidden_size=64, num_layers=2, output_len=180):
        super(LSTMAutoencoder, self).__init__()
        
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.output_len = output_len
        
        # Encoder
        self.encoder = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.2
        )
        
        # Decoder
        self.decoder = nn.LSTM(
            input_size=hidden_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.2
        )
        
        # Capa de salida
        self.fc = nn.Linear(hidden_size, 1)
        
    def forward(self, x):
        # x shape: (batch, seq_len, 1)
        
        # Encode
        _, (hidden, cell) = self.encoder(x)
        
        # Preparar entrada para decoder
        decoder_input = hidden[-1].unsqueeze(1).repeat(1, self.output_len, 1)
        
        # Decode
        decoder_output, _ = self.decoder(decoder_input, (hidden, cell))
        
        # Salida
        output = self.fc(decoder_output)
        
        return output.squeeze(-1)

# Configuración
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️ Usando dispositivo: {device}")

# Crear modelo
INPUT_LEN = X_train_norm.shape[1]  # 1440
OUTPUT_LEN = y_train_norm.shape[1]  # 180

model = LSTMAutoencoder(
    input_size=1,
    hidden_size=64,
    num_layers=2,
    output_len=OUTPUT_LEN
).to(device)

print(f"\n📐 Arquitectura del modelo:")
print(f"   Entrada: {INPUT_LEN} muestras (4 segundos)")
print(f"   Salida: {OUTPUT_LEN} muestras (0.5 segundos)")
print(f"   Parámetros totales: {sum(p.numel() for p in model.parameters()):,}")
print(model)

In [ ]:
# Configurar DataLoaders
BATCH_SIZE = 64
EPOCHS = 30
LEARNING_RATE = 0.001

# Crear tensores
X_train_tensor = torch.tensor(X_train_norm).unsqueeze(-1)  # (N, seq_len, 1)
y_train_tensor = torch.tensor(y_train_norm)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

# Optimizador y función de pérdida
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)

print(f"⚙️ Configuración de entrenamiento:")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Epochs: {EPOCHS}")
print(f"   Learning rate: {LEARNING_RATE}")
print(f"   Batches por epoch: {len(train_loader)}")

In [ ]:
# Entrenamiento
print("\n" + "="*60)
print("🚀 INICIANDO ENTRENAMIENTO")
print("="*60 + "\n")

train_losses = []
best_loss = float('inf')

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0
    
    for batch_X, batch_y in train_loader:
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)
        
        optimizer.zero_grad()
        
        # Forward
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        
        # Backward
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)
    scheduler.step(avg_loss)
    
    # Guardar mejor modelo
    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, 'best_model.pth'))
    
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1:3d}/{EPOCHS}] | Loss: {avg_loss:.6f} | LR: {optimizer.param_groups[0]['lr']:.6f}")

print(f"\n✅ Entrenamiento completado")
print(f"   Mejor loss: {best_loss:.6f}")

In [ ]:
# Visualizar curva de entrenamiento
plt.figure(figsize=(10, 4))
plt.plot(train_losses, 'b-', linewidth=2)
plt.title('📉 Curva de Entrenamiento', fontsize=12)
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. 📈 Evaluación del Modelo

Evaluamos el modelo usando el conjunto de test (pacientes nunca vistos durante el entrenamiento).

### Estrategia de Clasificación:
1. Calcular error de reconstrucción para cada latido
2. Establecer umbral basado en percentil del error en latidos normales
3. Clasificar como anomalía si el error supera el umbral

In [ ]:
# Cargar mejor modelo
model.load_state_dict(torch.load(os.path.join(OUTPUT_DIR, 'best_model.pth')))
model.eval()

# Preparar datos de test
X_test_tensor = torch.tensor(X_test_norm).unsqueeze(-1).to(device)
y_test_tensor = torch.tensor(y_test_norm).to(device)

# Predicción
with torch.no_grad():
    predictions = model(X_test_tensor)

# Calcular error de reconstrucción (MSE por muestra)
mse_per_sample = torch.mean((predictions - y_test_tensor) ** 2, dim=1).cpu().numpy()

print(f"📊 Estadísticas de error:")
print(f"   Media: {mse_per_sample.mean():.6f}")
print(f"   Std: {mse_per_sample.std():.6f}")
print(f"   Min: {mse_per_sample.min():.6f}")
print(f"   Max: {mse_per_sample.max():.6f}")

In [ ]:
# Análisis de errores por tipo de latido
df_results = pd.DataFrame({
    'label': labels_test,
    'mse': mse_per_sample,
    'is_normal': labels_test == 'N'
})

# Estadísticas por tipo
print("\n📊 Error promedio por tipo de latido:")
error_by_type = df_results.groupby('label')['mse'].agg(['mean', 'std', 'count']).sort_values('mean', ascending=False)
print(error_by_type.head(10))

In [ ]:
# Visualización de distribución de errores
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma de errores
normal_errors = mse_per_sample[labels_test == 'N']
abnormal_errors = mse_per_sample[labels_test != 'N']

axes[0].hist(normal_errors, bins=50, alpha=0.7, label='Normal', color='green', density=True)
axes[0].hist(abnormal_errors, bins=50, alpha=0.7, label='Arritmia', color='red', density=True)
axes[0].set_title('📊 Distribución de Error de Reconstrucción', fontsize=12)
axes[0].set_xlabel('MSE')
axes[0].set_ylabel('Densidad')
axes[0].legend()
axes[0].set_yscale('log')

# Box plot por tipo
df_box = df_results.copy()
df_box['tipo'] = df_box['is_normal'].map({True: 'Normal', False: 'Arritmia'})
sns.boxplot(data=df_box, x='tipo', y='mse', ax=axes[1])
axes[1].set_title('📊 Error por Categoría', fontsize=12)
axes[1].set_xlabel('Tipo')
axes[1].set_ylabel('MSE')

plt.tight_layout()
plt.show()

In [ ]:
# Determinar umbral óptimo
# Usamos el percentil 90 de los errores normales como umbral
normal_errors = mse_per_sample[labels_test == 'N']

# Probar diferentes percentiles
print("\n🎯 Búsqueda de umbral óptimo:")
print("-" * 60)

best_f1 = 0
best_threshold = 0

for percentile in [85, 90, 95, 97, 99]:
    threshold = np.percentile(normal_errors, percentile)
    
    # Clasificar
    y_true = (labels_test != 'N').astype(int)
    y_pred = (mse_per_sample > threshold).astype(int)
    
    # Métricas
    from sklearn.metrics import f1_score, precision_score, recall_score
    f1 = f1_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    
    print(f"Percentil {percentile}: Threshold={threshold:.6f} | F1={f1:.3f} | Precision={precision:.3f} | Recall={recall:.3f}")
    
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = threshold
        best_percentile = percentile

print(f"\n✅ Mejor umbral: {best_threshold:.6f} (percentil {best_percentile})")

In [ ]:
# Evaluación final con mejor umbral
threshold = best_threshold
y_true = (labels_test != 'N').astype(int)
y_pred = (mse_per_sample > threshold).astype(int)

print("\n" + "="*60)
print("📊 RESULTADOS FINALES")
print("="*60)
print(f"\nUmbral utilizado: {threshold:.6f}")
print("\n" + classification_report(y_true, y_pred, target_names=['Normal', 'Arritmia']))

# Matriz de confusión
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Normal', 'Arritmia'],
            yticklabels=['Normal', 'Arritmia'])
plt.title('📊 Matriz de Confusión', fontsize=12)
plt.xlabel('Predicción')
plt.ylabel('Real')
plt.tight_layout()
plt.show()

## 8. 💾 Exportación del Modelo

Exportamos el modelo entrenado y los parámetros necesarios para su uso en el sistema de streaming con Kafka.

In [ ]:
import pickle

# Guardar modelo completo
model_path = os.path.join(OUTPUT_DIR, 'ecg_forecaster.pth')
torch.save({
    'model_state_dict': model.state_dict(),
    'model_config': {
        'input_size': 1,
        'hidden_size': 64,
        'num_layers': 2,
        'output_len': OUTPUT_LEN,
        'input_len': INPUT_LEN
    },
    'threshold': threshold,
    'scaler_X_min': scaler_X.data_min_[0],
    'scaler_X_max': scaler_X.data_max_[0],
    'scaler_y_min': scaler_y.data_min_[0],
    'scaler_y_max': scaler_y.data_max_[0],
}, model_path)

print(f"✅ Modelo guardado en: {model_path}")

# Guardar scalers
scalers_path = os.path.join(OUTPUT_DIR, 'scalers.pkl')
with open(scalers_path, 'wb') as f:
    pickle.dump({'scaler_X': scaler_X, 'scaler_y': scaler_y}, f)
    
print(f"✅ Scalers guardados en: {scalers_path}")

# Información del modelo
print(f"\n📋 Información del modelo exportado:")
print(f"   Umbral de detección: {threshold:.6f}")
print(f"   Longitud de entrada: {INPUT_LEN} muestras (4 segundos a 360Hz)")
print(f"   Longitud de salida: {OUTPUT_LEN} muestras (0.5 segundos a 360Hz)")

In [ ]:
# Exportar también un archivo parquet con un ejemplo para testing
# Seleccionar un paciente de test para exportar
sample_patient = test_patients[0]
df_sample = df_completo[df_completo['record_id'] == sample_patient].copy()

# Obtener columna de señal
signal_cols = [c for c in df_sample.columns if c not in ['record_id', 'fs', 'sample_idx', 'annotation']]
signal_col = signal_cols[0]

# Preparar DataFrame simplificado
df_export = df_sample[[signal_col, 'annotation']].copy()
df_export.columns = ['signal', 'annotation']

# Guardar en directorio src
parquet_path = os.path.join(BASE_DIR, 'src', f'{sample_patient}.parquet')
df_export.to_parquet(parquet_path)

print(f"✅ Archivo de ejemplo guardado en: {parquet_path}")
print(f"   Paciente: {sample_patient}")
print(f"   Muestras: {len(df_export):,}")

## 📝 Conclusiones

### Resultados Obtenidos:

1. **Preprocesamiento**: Se segmentaron exitosamente los latidos del ECG en ventanas de 4.5 segundos

2. **Prevención de Data Leakage**: La división train/test se realizó a nivel de paciente, garantizando que no haya contaminación entre conjuntos

3. **Modelo**: El LSTM Autoencoder aprendió a predecir patrones normales de ECG

4. **Detección de Anomalías**: Se estableció un umbral óptimo basado en el error de reconstrucción

### Uso del Modelo:

El modelo exportado puede utilizarse en el sistema de streaming con Kafka para:
- Recibir señales ECG en tiempo real
- Predecir el siguiente segmento
- Detectar arritmias cuando el error supera el umbral

### Próximos Pasos:
- Integrar el modelo en `test_sine_plotter.py`
- Probar con streaming en tiempo real
- Ajustar umbral según necesidades clínicas

---
*Proyecto desarrollado para Big Data - Universidad*